# Night Flash — Google Colab

Six cells. Everything after cell 2 is identical to the Kaggle notebook.

| cell | what it does |
|---|---|
| 1 | install and import |
| 2 | mount Drive, point at the data |
| 3 | **configuration — one object** |
| 4 | **briefing — read this before training** |
| 5 | train / infer |
| 6 | export and download |

The TUI does not run in Colab. `nightflash-tui` is for your Mac.

In [ ]:
# ── 1. install ──────────────────────────────────────────────────────
import subprocess, sys, shutil, os
from pathlib import Path

REPO_URL = ""      # optional: set to a git URL, otherwise upload the zip
PROJECT = Path("/content/nightflash-edge")

if REPO_URL:
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
else:
    if not (PROJECT / "pyproject.toml").exists():
        zips = sorted(Path("/content").glob("*.zip"))
        if not zips:
            raise RuntimeError(
                "Upload dist/nightflash-edge-colab.zip to /content/ via the file "
                "browser, or set REPO_URL at the top of this cell."
            )
        subprocess.run(["unzip", "-q", str(zips[0]), "-d", "/content"], check=True)

assert (PROJECT / "pyproject.toml").exists()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / "src"))

import torch
from nightflash import Session
from nightflash.briefing import brief
from nightflash.notebook import show_previews, show_image
from nightflash.kaggle import environment_report

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu   ", torch.cuda.get_device_name(0))

In [ ]:
# ── 2. data ────────────────────────────────────────────────────────
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/nightflash-edge")  # edit if different
DATASET = DRIVE_ROOT / "training_photos"
RUNS = DRIVE_ROOT / "runs"

SOURCE_DIR = DATASET / "normal_photos"
TARGET_DIR = DATASET / "disposable_photos"
VAL_DIR = DATASET / "validation_photos"
MASK_DIR = None  # set to DATASET/"source_masks" when you have Vision masks

for label, path in [("source", SOURCE_DIR), ("target", TARGET_DIR)]:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} not found: {path}")
    n = sum(1 for p in path.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    print(f"{label:7s} {n:6d} images")
print("runs root:", RUNS)

In [ ]:
# ── 3. configuration — the only thing you normally edit ─────────────
s = Session(
    arch="edgehat",          # edgehat | swinir | restormer | hybrid
    head="freqsplit",        # freqsplit | lut3d | perpixel | tanh
    name="night",            # run folder suffix: edgehat_freqsplit_<name>
    source_dir=SOURCE_DIR,
    target_dir=TARGET_DIR,
    val_dir=VAL_DIR,
    mask_dir=MASK_DIR,
    runs_root=RUNS,
    image_size=256,
    batch_size=8,
    epochs=50,                # absolute target, not extra epochs
    device="auto",
    use_amp=True,
    num_workers=2,
    preview_every=250,
    log_every=25,
    halation=True,
    subject_gate=True,
    region_exposure=True,
)
print(s)

In [ ]:
# Pick ONE. Each line is idempotent and safe to re-run.

s.train()                                   # resume, or start fresh
# s.train(extra_epochs=50)                  # resume and add 50 more
# s.infer(f"{SOURCE_DIR}", seeds=3)        # inference only
# s.infer(photo, tile=512)                  # full-resolution detail
# show_previews(s.run_dir, n=6)

# Heads: tanh | perpixel | freqsplit | lut3d
#        lut3d+freqsplit | lut3d+perpixel | tanh+freqsplit


In [ ]:
# Pick ONE. Each line is idempotent and safe to re-run.

s.train()                                   # resume, or start fresh
# s.train(extra_epochs=50)                  # resume and add 50 more
# s.infer(f"{SOURCE_DIR}", seeds=3)        # inference only
# s.infer(photo, tile=512)                  # full-resolution detail
# show_previews(s.run_dir, n=6)

# Heads: tanh | perpixel | freqsplit | lut3d
#        lut3d+freqsplit | lut3d+perpixel | tanh+freqsplit


In [ ]:
# ── 6. export and collect ──────────────────────────────────────────
from nightflash.export_coreml import build_parser as _ep, export as _export
import argparse as _ap

if s.latest.exists():
    _a = _ep().parse_args([
        "--checkpoint", str(s.latest),
        "--output", str(s.run_dir / "NightFlashEdge.mlpackage"),
        "--image-size", str(s.image_size),
        "--half",
    ])
    try:
        _export(_a)
    except Exception as exc:
        print("Core ML export needs macOS + coremltools; download the .pt instead:")
        print("  ", s.latest)

print("\nRun directory:", s.run_dir)
print("State:", s.state().status)

show_previews(s.run_dir, n=4)